In [ ]:
import numpy as np
import pandas as pd

# =============================================================================
# 1. LOAD INPUT DATA
# =============================================================================

try:
    df = pd.read_csv("Luhman_sed.csv")
except FileNotFoundError:
    print("ERROR: 'Luhman_sed.csv' not found.")
    raise SystemExit

# Keep only required columns
df = df.dropna(subset=["band", "app_magnitude"]).copy()
df = df[["band", "app_magnitude", "app_magnitude_unc"]]

# =============================================================================
# 2. DELTA-MAGNITUDE TABLE
#    Format: delta_m = m_B - m_A
#             uncertainty in delta_m
# =============================================================================

delta_m_info = {
    "2MASS.J":  (-0.35, 0.03),
    "2MASS.H":  ( 0.00, 0.03),
    "2MASS.Ks": ( 0.10, 0.03),

    "WFCAM.Y": (-0.15, 0.03),
    "WFCAM.J": (-0.35, 0.03),
    "WFCAM.H": ( 0.00, 0.03),
    "WFCAM.K": ( 0.10, 0.03),

    "IRAC.I1": ( 0.05, 0.05),
    "IRAC.I2": ( 0.08, 0.05),

    "WISE.W1": ( 0.05, 0.05),
    "WISE.W3": ( 0.12, 0.08),
    "WISE.W4": ( 0.15, 0.10),
}

# =============================================================================
# 3. MAP DELTA-M VALUES
# =============================================================================

df["delta_m"] = df["band"].map(
    lambda x: delta_m_info[x][0] if x in delta_m_info else np.nan
)

df["delta_m_unc"] = df["band"].map(
    lambda x: delta_m_info[x][1] if x in delta_m_info else np.nan
)

# Check for missing filters
missing = df[df["delta_m"].isna()]

if len(missing) > 0:
    print("\nERROR: Missing delta_m values for:")
    print(missing["band"].unique())
    raise ValueError("Provide delta_m values for all filters.")

# =============================================================================
# 4. DECONVOLUTION
# =============================================================================

m_tot = df["app_magnitude"].values
sigma_tot = df["app_magnitude_unc"].values

delta_m = df["delta_m"].values
sigma_delta = df["delta_m_unc"].values

# Flux ratio
r = 10**(-0.4 * delta_m)

# Component magnitudes
mag_A = m_tot + 2.5 * np.log10(1 + r)
mag_B = mag_A + delta_m

# =============================================================================
# 5. UNCERTAINTY PROPAGATION
# =============================================================================

# Derivative of m_A with respect to delta_m
dmA_ddm = -r / (1 + r)

sigma_A = np.sqrt(
    sigma_tot**2 +
    (dmA_ddm * sigma_delta)**2
)

sigma_B = np.sqrt(
    sigma_A**2 +
    sigma_delta**2
)

# Store results
df["mag_A"] = mag_A
df["mag_B"] = mag_B

df["mag_A_unc"] = sigma_A
df["mag_B_unc"] = sigma_B

# =============================================================================
# 6. FLUX CONSERVATION CHECK
# =============================================================================

F_total = 10**(-0.4 * m_tot)
F_A = 10**(-0.4 * mag_A)
F_B = 10**(-0.4 * mag_B)

max_flux_error = np.max(np.abs(F_total - (F_A + F_B)))

print(f"\nMaximum flux conservation error = {max_flux_error:.3e}")

# =============================================================================
# 7. BUILD OUTPUT TABLES
# =============================================================================

df_A = pd.DataFrame({
    "band": df["band"],
    "app_magnitude": np.round(df["mag_A"], 3),
    "app_magnitude_unc": np.round(df["mag_A_unc"], 3)
})

df_B = pd.DataFrame({
    "band": df["band"],
    "app_magnitude": np.round(df["mag_B"], 3),
    "app_magnitude_unc": np.round(df["mag_B_unc"], 3)
})

# =============================================================================
# 8. SAVE OUTPUT
# =============================================================================

df_A.to_csv("Luhman_16A_unblended.csv", index=False)
df_B.to_csv("Luhman_16B_unblended.csv", index=False)

print("\nExtraction Complete!")
print("Saved: Luhman_16A_unblended.csv")
print("Saved: Luhman_16B_unblended.csv")

# =============================================================================
# 9. PREVIEW
# =============================================================================

print("\nLuhman 16A:")
print(df_A.head())

print("\nLuhman 16B:")
print(df_B.head())